# ML-08 — Capstone Modeling Lane

This notebook builds a honest model for the weekly industry‑brief task and compares it to the rule‑based baseline from Week‑4. All steps follow the same data, split, and metric so the comparison is fair.


## 1. Method choice and why

We start with a **Logistic Regression** because the baseline is a rule‑based binary classifier and LR gives us interpretable coefficients while being competitive. If we need extra power we can switch to a **Random Forest** (toggle `MODEL_TYPE`).


In [ ]:
# Choose model type
MODEL_TYPE = "logreg"  # options: 'logreg' or 'rf'


## 2. Split design

We replicate the Week‑4 hold‑out split: 70 % train, 30 % test, seeded with 42 for reproducibility. This matches the baseline split so metrics are comparable.


In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# Load data – same file used for baseline
data_path = 'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(data_path)

# Basic preprocessing – mirror baseline's features
# Example: create binary label 'stale_flag' (1 if days_since_last_update >= 90)
df['stale_flag'] = (df['days_since_last_update'] >= 90).astype(int)
# Use impressions_90d (log‑scaled) as a feature
df['log_impr'] = np.log1p(df['impressions_90d'])
# Simple feature set
X = df[['stale_flag', 'log_impr']].values
y = df['stale_flag'].values  # same target used in baseline rule

# Train‑test split (same random_state as baseline)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# Save split indices for reproducibility (optional)
np.save('train_idx.npy', X_train)
np.save('test_idx.npy', X_test)


## 3. Train + compare vs baseline

We train the chosen model, compute prediction scores, and evaluate precision@20 and precision@50 – the same metrics reported for the baseline.


In [ ]:
def precision_at_k(y_true, scores, k):
    # Sort by descending score
    order = np.argsort(-scores)
    top_k = order[:k]
    return np.mean(y_true[top_k])

# Train model
if MODEL_TYPE == 'logreg':
    model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
else:
    model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

# Scores – probability of the positive class
train_scores = model.predict_proba(X_train)[:, 1]
test_scores = model.predict_proba(X_test)[:, 1]

# Load baseline scores that were saved during Week‑4 (they live in work/outputs/baseline_action_score.csv)
baseline_path = 'work/outputs/baseline_action_score.csv'
baseline_df = pd.read_csv(baseline_path)
# Align baseline rows with our test set using the same index ordering (baseline already used the same split)
baseline_scores = baseline_df.loc[baseline_df['split']=='test', 'baseline_score'].values
baseline_labels = baseline_df.loc[baseline_df['split']=='test', 'label'].values

# Compute precision@20 and @50 for both
k_vals = [20, 50]
results = []
for k in k_vals:
    model_prec = precision_at_k(y_test, test_scores, k)
    baseline_prec = precision_at_k(baseline_labels, baseline_scores, k)
    results.append({'k': k, 'model': model_prec, 'baseline': baseline_prec})

# Display table
import pandas as pd
result_df = pd.DataFrame(results)
display(result_df)


## 4. Errors and interpretation

We look at the top‑5 false‑positive and false‑negative cases to understand where the model struggles, and we show feature importances (or coefficients) for interpretability.


In [ ]:
# Identify false positives / false negatives on test set
pred_labels = (test_scores >= 0.5).astype(int)
fp_idx = np.where((pred_labels == 1) & (y_test == 0))[0]
fn_idx = np.where((pred_labels == 0) & (y_test == 1))[0]

print(f'False positives: {len(fp_idx)}, false negatives: {len(fn_idx)}')

# Show a few examples (first 3 of each)
def show_examples(idxs, name):
    print(f'--- {name} examples ---')
    for i in idxs[:3]:
        row = X_test[i]
        print(f'Features: stale={row[0]}, log_impr={row[1]:.2f}, true={y_test[i]}, score={test_scores[i]:.3f}')

show_examples(fp_idx, 'False Positive')
show_examples(fn_idx, 'False Negative')

# Feature importance / coefficients
if MODEL_TYPE == 'logreg':
    coef = model.coef_[0]
    feat_names = ['stale_flag', 'log_impr']
    importance = pd.Series(coef, index=feat_names).sort_values(ascending=False)
else:
    importance = pd.Series(model.feature_importances_, index=['stale_flag', 'log_impr']).sort_values(ascending=False)
print('
Feature importance / coefficients:')
display(importance)


## Self‑check

- [ ] Every markdown section is filled.
- [ ] All code cells run without error (Run All).
- [ ] No private URLs or client identifiers appear.
- [ ] Claims are phrased carefully (observed, measured).
- [ ] Notebook committed under `work/notebooks/`.
